# MusPsy Model Fine-Tuning Guide on RunPod

This notebook contains the complete pipeline to set up and run the multi-task fine-tuning for **MusPsy** using **LLaMA-Factory** on RunPod.

## Step 0: Check CUDA works before installing anything
Cheap to check now (seconds), expensive to discover after a multi-minute install. This session hit the same failure repeatedly on certain Community Cloud pods: `nvidia-smi` reports a perfectly healthy GPU, but `torch.cuda.is_available()` returns `False` - traced to `CUDA_VISIBLE_DEVICES` being set to an empty string, which hides all devices from the CUDA runtime (a different code path than `nvidia-smi`'s, which uses NVML and isn't affected). Secure Cloud pods resolved it reliably; if this check fails, try a different pod before spending time on Steps 1-2.

In [ ]:
!nvidia-smi

try:
    import torch
    print("torch already installed:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    assert torch.cuda.is_available(), (
        "CUDA not available even though nvidia-smi above may show a healthy GPU. If "
        "CUDA_VISIBLE_DEVICES is set to an empty string, that's the cause (echo it to check). "
        "Try a different pod - Secure Cloud resolved this reliably - before proceeding to Steps 1-2."
    )
    print("CUDA check passed - safe to proceed to Step 1.")
except ImportError:
    print("torch not installed yet on this pod - will be installed fresh in Step 2 "
          "(`uv pip install torch==2.8.0 ...`). Re-run this cell after Step 2 to confirm "
          "CUDA works before launching training.")

## Step 1: Clone the Repositories
First, we clone the **MusPsy** repository (to get the training data) and the **LLaMA-Factory** repository (for the training framework).

In [ ]:
# Clone MusPsy repository
!git clone https://github.com/Rener2005/MusPsy.git

# Clone LLaMA-Factory
!git clone https://github.com/hiyouga/LLaMA-Factory.git

## Step 2: Install Dependencies
Install the required libraries for LLaMA-Factory and align package versions to prevent PyTorch conflicts.

In [ ]:
%cd LLaMA-Factory

# Explicit torch pin BEFORE the LLaMA-Factory install below - RTX 5090 (Blackwell,
# sm_120) needs a torch build with matching CUDA-capability support, which only
# shipped as of torch 2.8's cu128 wheels; torch 2.7 and earlier have no Blackwell
# support at all. Without this pin, `pip install -e .[torch,metrics]` would just
# resolve to whatever's already on the pod (which may predate 2.8) rather than
# forcing the version this GPU actually requires. Not needed in muspsy_serve_vllm.ipynb
# - serving still runs on an older-generation GPU, so that notebook's vllm==0.9.2 /
# torch==2.7.0 pin (chosen to dodge a separate vLLM packaging bug) stays as-is.
#
# torchaudio==2.8.0 is pinned alongside torch here too - LLaMA-Factory's own
# data/mm_plugin.py imports torchaudio directly (multimedia support), and torchaudio's
# own versioning tracks torch's 1:1 since the 2.x line, so 2.8.0 is the correct pair.
#
# --break-system-packages on every pip/uv call below: this pod's Python 3.12 base
# image marks itself "externally managed" (PEP 668), which blocks --system/plain
# installs by default - a newer-base-image behavior not seen on the earlier
# (pre-Blackwell) pods this project used. Safe to override here: the pod is an
# ephemeral, single-purpose container with no separate system Python environment
# worth protecting. uv's flag name matches pip's own (`--break-system-packages`).
#
# `pip install -U uv` FIRST is required, not optional - a real run on a fresh pod hit
# `/bin/bash: line 1: uv: command not found` on the `uv pip install` line below, which
# (since Jupyter's `!` doesn't raise a Python exception for a failed shell command) let
# execution silently continue with the pod's pre-existing torch (2.4.1+cu124 in that
# case) instead of the pinned 2.8.0 - only caught downstream by the assert. uv isn't
# guaranteed pre-installed on every pod image (it was on the ones muspsy_serve_vllm*.ipynb
# happened to run on, but not this one), so bootstrap it explicitly rather than assume.
!pip install -U uv --break-system-packages
!uv pip install --system --break-system-packages --reinstall "torch==2.8.0" "torchaudio==2.8.0" --index-url https://download.pytorch.org/whl/cu128

# If Step 0's GPU check already ran in THIS kernel (it does `import torch` to read
# torch.cuda.is_available()), `torch` is already sitting in sys.modules. Python caches
# imports by module name - `import torch` below would just return that SAME cached
# module object and its OLD __version__, even though the reinstall above genuinely
# replaced the package on disk (confirmed once by a real run: `uv`'s own diff output
# showed "- torch==2.4.1+cu124 / + torch==2.8.0+cu128", yet the following `import torch`
# still reported 2.4.1 and the version assert failed). This is not a failed install -
# restarting the kernel and re-running from this cell is the actual fix, not re-running
# the install again.
import sys
if "torch" in sys.modules:
    print(
        "NOTE: 'torch' was already imported in this kernel (most likely by the Step 0 GPU "
        "check) before this reinstall. The `import torch` below will return that SAME "
        "cached module and its OLD version, regardless of what was just installed on disk. "
        "If the version assert right after this fails, RESTART THE KERNEL and re-run from "
        "this Step 2 cell - do not just re-run the install."
    )

import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available - check nvidia-smi and that this pod actually has a Blackwell (RTX 5090) GPU attached."
from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.8.0"), (
    f"torch pin failed, got {torch.__version__!r}. If the `uv pip install` output above "
    f"actually shows torch being upgraded (a '- torch==... / + torch==2.8.0+...' diff line), "
    f"this is almost certainly the sys.modules caching issue described above, not a real "
    f"install failure - restart the kernel and re-run from this cell."
)

!pip install --break-system-packages -e .[torch,metrics]

# transformers==5.8.0 (previously pinned to 4.57.6 - see below for why this changed).
#
# The 4.x line NEVER shipped support for Qwen3.5's "qwen3_5" model type at all (confirmed
# directly: the qwen3_5 model module is a 404 at the v4.57.6 git tag). It only exists from
# v5.8.0 onward (confirmed: 404 at v5.0.0, present at v5.8.0) - so training the experimental
# qwen3.5-9b option in Step 4 genuinely requires 5.x, not a config/pin mistake to route around.
#
# This notebook originally avoided ALL of transformers 5.x specifically because 5.x
# unconditionally imports a torch.library.custom_op-based MoE/fp8 module at import time
# (integrations/moe.py) that OLDER/mismatched torch builds can't schema-infer, crashing with
# a misleading "unsupported type torch.Tensor" error. That reasoning was written when this
# pod's torch was whatever was pre-installed (2.4.x-era) - now that torch is explicitly pinned
# above to a modern, matched 2.8.0+cu128 build, the crash may simply not reproduce anymore.
# This is a real, live experiment, not a settled fact: if `import transformers` below crashes
# with that "unsupported type torch.Tensor" error, the moe.py issue is confirmed to still
# apply even on torch 2.8.0, and this pin should be reverted to 4.57.6 (which means abandoning
# the qwen3.5-9b option, since 4.x can't recognize it regardless).
#
# 5.8.0 specifically (not the newest 5.x) because it's the exact ceiling LLaMA-Factory's own
# dependency constraints allow.
#
# peft==0.18.1 / accelerate==1.11.0 are UNCHANGED from the 4.57.6-era pins - not verified
# against transformers 5.8.0 specifically. If either import fails or behaves oddly after this
# bump, that's the next thing to check (their own dependency floors may have moved for 5.x).
!pip install --break-system-packages "transformers==5.8.0" "peft==0.18.1" "accelerate==1.11.0"
!pip install --break-system-packages bitsandbytes huggingface_hub

# Re-pin torch/torchaudio AGAIN, after the LLaMA-Factory extras install - not redundant.
# A real run showed `pip install -e .[torch,metrics]` above silently re-resolves torch AND
# torchaudio from default PyPI (no --index-url there), clobbering the cu128 pin set earlier
# in this same cell: the pod ended up with torch reporting CUDA 12.1 and torchaudio reporting
# CUDA 12.4 - neither matching our cu128 pin, and mismatched with EACH OTHER - causing
# `RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA
# versions` the moment LLaMA-Factory's own data/mm_plugin.py did `import torchaudio` at
# training launch. Forcing the pin back here, after the extras install, is what actually
# guarantees a consistent pair survives to the training launch in Step 5.
!uv pip install --system --break-system-packages --reinstall "torch==2.8.0" "torchaudio==2.8.0" --index-url https://download.pytorch.org/whl/cu128

# Verify the pins actually took effect before proceeding. NOTE: if this kernel already had
# `torch`/`torchaudio` imported earlier in this same cell run (they were, just above), Python's
# module cache means re-importing here won't reflect this second reinstall either - the prints
# below will still show whatever was loaded by the FIRST `import torch` in this cell. A version
# mismatch here isn't necessarily a real problem in that case; what actually matters is what
# gets loaded fresh when `llamafactory-cli train` launches as its OWN separate process in
# Step 5 (a clean process, no stale sys.modules) - trust Step 5b's log tail as the real signal,
# not this cell's printed version if you're unsure.
import transformers, peft, accelerate
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
assert transformers.__version__ == "5.8.0", f"transformers pin failed, got {transformers.__version__}"
assert peft.__version__ == "0.18.1", f"peft pin failed, got {peft.__version__}"

## Step 3: Regenerate corrected data and register MusPsy Datasets in LLaMA-Factory
`train/task3.json` (as published) has two labeling bugs - see `MusPsy/task3_data_quality_finding.md`
for the full analysis: (1) `input`/`output` are frequently a same-speaker pair rather than a genuine
client→counselor exchange, and (2) `history`'s pairing is reversed for ~25% of records. Both are
corrected here directly from `train/task1.json`'s ground-truth labeled transcripts, regenerating
`train/task3_fixed.json` fresh on this pod (rather than depending on that file having made it into
the cloned git repo). Then we register `task1`, `task2`, `task3` (as published), and `task3_fixed`
with LLaMA-Factory - Step 4's `DATASET_VARIANT` toggle picks which `task3` variant a given training
run actually uses.

In [ ]:
# --- Regenerate train/task3_fixed.json from ground truth (see MusPsy/task3_data_quality_finding.md
# and MusPsy/fix_task3_labels.py for the full writeup) - embedded here so this notebook doesn't
# depend on task3_fixed.json having been committed to the cloned MusPsy git repo. ---

import json
from pathlib import Path

muspsy_train_dir_path = Path("../MusPsy/train").resolve()
SNIPPET_LEN = 30
SYNTHETIC_OPENER = "Hi, I'm Here Today."


def _parse_transcript(text):
    """Parse a Consultant:/User:-labeled transcript into (role, text) tuples."""
    segments = [s.strip() for s in text.strip().split("\n\n") if s.strip()]
    turns = []
    for seg in segments:
        if seg.startswith("Consultant:"):
            turns.append(("Consultant", seg[len("Consultant:"):].strip()))
        elif seg.startswith("User:"):
            turns.append(("User", seg[len("User:"):].strip()))
        elif turns:
            role, prev_text = turns[-1]
            turns[-1] = (role, f"{prev_text}\n{seg}")
    return turns


def _find_anchor(turns, history):
    """Locate history's last pair inside the transcript, checking both possible
    element orderings (normal client->counselor, or reversed counselor->client)."""
    if not history:
        return 0
    a, b = history[-1]
    snip_a, snip_b = a.strip()[:SNIPPET_LEN], b.strip()[:SNIPPET_LEN]
    for i in range(len(turns) - 1):
        role_x, text_x = turns[i]
        role_y, text_y = turns[i + 1]
        if role_x == "User" and text_x.startswith(snip_a) and role_y == "Consultant" and text_y.startswith(snip_b):
            return i + 2
        if role_x == "Consultant" and text_x.startswith(snip_a) and role_y == "User" and text_y.startswith(snip_b):
            return i + 2
    return None


def _rebuild_history(turns, end):
    """Rebuild correctly-ordered (client, counselor) history from turns[0:end].
    96.4% of sessions genuinely open with the counselor speaking first, so the
    unpaired opening counselor turn is paired with the same synthetic client
    placeholder the original dataset's own construction already uses in 73.6%
    of records, rather than dropping real opening-greeting training signal."""
    pairs = []
    i = 0
    if end > 0 and turns[0][0] == "Consultant":
        pairs.append([SYNTHETIC_OPENER, turns[0][1]])
        i = 1
    while i < end - 1:
        role_a, text_a = turns[i]
        role_b, text_b = turns[i + 1]
        if role_a == "User" and role_b == "Consultant":
            pairs.append([text_a, text_b])
            i += 2
        else:
            i += 1
    return pairs


def regenerate_task3_fixed(train_dir: Path) -> int:
    task1 = json.loads((train_dir / "task1.json").read_text(encoding="utf-8"))
    task3 = json.loads((train_dir / "task3.json").read_text(encoding="utf-8"))

    fixed = []
    for i, rec3 in enumerate(task3):
        if i >= len(task1) or not task1[i].get("input"):
            continue
        history = rec3.get("history", [])
        turns = _parse_transcript(task1[i]["input"])
        anchor = _find_anchor(turns, history)
        if anchor is None:
            continue

        pair_start = None
        for j in range(anchor, len(turns) - 1):
            if turns[j][0] == "User" and turns[j + 1][0] == "Consultant":
                pair_start = j
                break
        if pair_start is None:
            continue

        role_in, text_in = turns[pair_start]
        role_out, text_out = turns[pair_start + 1]
        if role_in != "User" or role_out != "Consultant":
            continue

        new_rec = dict(rec3)
        new_rec["history"] = _rebuild_history(turns, pair_start)
        new_rec["input"] = text_in
        new_rec["output"] = text_out
        fixed.append(new_rec)

    out_path = train_dir / "task3_fixed.json"
    out_path.write_text(json.dumps(fixed, indent=4, ensure_ascii=False), encoding="utf-8")
    return len(fixed)


n_fixed = regenerate_task3_fixed(muspsy_train_dir_path)
print(f"Regenerated task3_fixed.json: {n_fixed} corrected records written to {muspsy_train_dir_path / 'task3_fixed.json'}")

# --- Register datasets with LLaMA-Factory ---

import os

dataset_info_path = "data/dataset_info.json"

# Load existing dataset info
with open(dataset_info_path, "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

# Absolute path to the MusPsy train directory
muspsy_train_dir = os.path.abspath("../MusPsy/train")

# Define the new dataset entries
muspsy_entries = {
    "muspsy_task1": {
        "file_name": os.path.join(muspsy_train_dir, "task1.json"),
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "history": "history"
        }
    },
    "muspsy_task2": {
        "file_name": os.path.join(muspsy_train_dir, "task2.json"),
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "history": "history"
        }
    },
    "muspsy_task3": {
        "file_name": os.path.join(muspsy_train_dir, "task3.json"),
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "history": "history"
        }
    },
    # Corrected version of task3.json - see MusPsy/task3_data_quality_finding.md.
    # Registered under a separate name (not overwriting "muspsy_task3") so both
    # the original and corrected data stay available - Step 4's DATASET_VARIANT
    # toggle picks which one a given training run uses, letting you train both
    # an "as-published" and a "corrected" adapter for a direct comparison.
    "muspsy_task3_fixed": {
        "file_name": os.path.join(muspsy_train_dir, "task3_fixed.json"),
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "history": "history"
        }
    }
}

# Update and save
dataset_info.update(muspsy_entries)
with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print("Registered MusPsy datasets successfully!")

## Step 4: Write Training Configuration File
We will write the training configuration file `muspsy_lora.yaml` dynamically.

* **Option A:** Optimized for **A100 (80GB)**. Trains at full **4096** context with batch size 4 for maximum speed (no gradient checkpointing needed).
* **Option B:** Optimized for **RTX 4090 (24GB)**. Trains at **3072** context with batch size 1 and gradient checkpointing.
* **Option C:** QLoRA 4-bit for budget GPUs (12-16GB VRAM).

In [ ]:
# "original" trains on task3.json exactly as published (the "as-published" baseline
# adapter, already trained: thanaphatt1/llama3-8b-muspsy). "fixed" trains on
# task3_fixed.json (see MusPsy/task3_data_quality_finding.md) for a direct
# before/after comparison. output_dir and HF_HUB_MODEL_ID below are derived from
# this so the two runs never overwrite each other or collide on the Hub.
DATASET_VARIANT = "fixed"  # "original" or "fixed"
assert DATASET_VARIANT in ("original", "fixed")
_task3_dataset_name = "muspsy_task3" if DATASET_VARIANT == "original" else "muspsy_task3_fixed"

# Backbone choice. "llama3-8b" is the paper's own backbone (Section 6.1) and what
# thanaphatt1/llama3-8b-muspsy-fixed was already trained on. "qwen3-8b" swaps to
# Qwen3-8B (Qwen3ForCausalLM, plain text-only causal LM) - picked after Qwen3.5-9B
# turned out to have three separate, serious problems for SERVING this exact use case:
# (1) Community Cloud GPU-passthrough unreliability (a RunPod issue, not model-
# specific, but compounded testing time), (2) an open, unresolved vLLM bug
# (github.com/vllm-project/vllm#49354, filed 2026-07-21) where --enable-lora
# silently does not apply LoRA adapters on Qwen3.5/3.6's hybrid-GDN architecture -
# would have silently served the un-fine-tuned base model with no error, and
# (3) Qwen3.5-9B's weights + hybrid-GDN state cache + torch.compile overhead alone
# nearly filled a 24GB card, leaving ~0 room for KV cache even for a single request.
# Plain Qwen3-8B has none of these: standard architecture (no hybrid-GDN, no VLM
# plugin), and no LoRA-specific bug reports turned up for it specifically.
#
# "qwen3.5-9b" (https://huggingface.co/Qwen/Qwen3.5-9B) is included below as a THIRD,
# EXPERIMENTAL option for training specifically - the vLLM-serving problems above don't
# apply to LLaMA-Factory's training stack at all (those were vLLM/inference-only bugs),
# but a check of LLaMA-Factory's own issue tracker turned up real, training-specific risk
# signals for this exact architecture:
#   - Confirmed and fixed in this notebook: the "qwen3_5" model type genuinely doesn't
#     exist in ANY transformers 4.x release (404 at the v4.57.6 git tag), only from v5.8.0
#     onward - Step 2's transformers pin was bumped from 4.57.6 to 5.8.0 specifically to
#     address this (real error hit and resolved: `ValueError: ... model type qwen3_5 ...
#     Transformers does not recognize this architecture`).
#   - hiyouga/LLaMA-Factory#10539: `ValueError: Processor was not found` fine-tuning Qwen3.5 -
#     plausible here too, since Qwen3.5 is a VLM-class checkpoint (Qwen3_5ForConditionalGeneration)
#     and this run is text-only SFT with no vision processor config. Not yet hit or ruled out.
#   - hiyouga/LLaMA-Factory#10530: reported structural token-sequence inconsistencies (4 of them)
#     between the qwen3_5/qwen3_6 template's training-time tokenization and vLLM's inference-time
#     tokenization - exactly the class of subtle train/serve mismatch that produced the earlier
#     mysterious Task 1 format-collapse on Qwen3-8B (see the file for that story). Not yet hit.
#   - hiyouga/LLaMA-Factory#10580, #10591: other unresolved training failures/OOM reports across
#     the Qwen3.5 family (different model sizes, so not guaranteed to reproduce on the 9B here).
#   - Step 2's transformers==5.8.0 bump is ALSO itself an experiment: this notebook originally
#     avoided all of 5.x due to a moe.py MoE/fp8 import crash on older/mismatched torch builds -
#     see Step 2's comments for the full reasoning on why that may no longer apply now that
#     torch is pinned to a modern 2.8.0+cu128 build, and what it means if it still does.
# None of the remaining risks above are confirmed to reproduce with our exact recipe - they
# just mean this option should be treated as an experiment to keep validating (watch Step 5b's
# log tail closely on each new run), not a drop-in equal alternative to llama3-8b/qwen3-8b yet.
BASE_MODEL_CHOICE = "qwen3-8b"  # "llama3-8b", "qwen3-8b", or "qwen3.5-9b" (experimental, see above)
assert BASE_MODEL_CHOICE in ("llama3-8b", "qwen3-8b", "qwen3.5-9b")
_BASE_MODEL_CONFIGS = {
    "llama3-8b": {"model_name_or_path": "meta-llama/Meta-Llama-3-8B-Instruct", "template": "llama3"},
    # qwen3_nothink (not qwen3, which is ReasoningTemplate-based and expects/produces
    # <think>...</think> blocks) - MusPsy's training data (task1/2/3.json) is plain
    # instruction->response with no reasoning traces, and Task 3 is meant to run as a
    # real-time counselor turn in a live multi-session simulation, not with chain-of-
    # thought latency added to every response.
    "qwen3-8b": {"model_name_or_path": "Qwen/Qwen3-8B", "template": "qwen3_nothink"},
    # qwen3_5_nothink for the same no-reasoning-traces reason as qwen3-8b above. Confirmed
    # registered in LLaMA-Factory's template.py (both "qwen3_5" and "qwen3_5_nothink" exist),
    # but see the big comment block above BASE_MODEL_CHOICE for real, open risk signals
    # specific to actually training this checkpoint - not yet fully validated in this project.
    "qwen3.5-9b": {"model_name_or_path": "Qwen/Qwen3.5-9B", "template": "qwen3_5_nothink"},
}
_base_cfg = _BASE_MODEL_CONFIGS[BASE_MODEL_CHOICE]

_output_dir = f"saves/{BASE_MODEL_CHOICE}/lora/muspsy-{DATASET_VARIANT}"

yaml_content_a100 = f"""\
### model
model_name_or_path: {_base_cfg["model_name_or_path"]}

### method
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 32
lora_alpha: 16

### dataset
dataset: muspsy_task1,muspsy_task2,{_task3_dataset_name}
template: {_base_cfg["template"]}
cutoff_len: 4096
max_samples: 100000
val_size: 0.05
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: {_output_dir}
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 6
gradient_accumulation_steps: 3
learning_rate: 0.0002
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
flash_attn: auto

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 500

### save (best-checkpoint selection, not just "whatever's left when training stops")
# save_steps (500, in ### output above) already matches eval_steps (500) - required for
# load_best_model_at_end: HF's Trainer needs save/eval strategy+interval to align exactly
# so it can match each saved checkpoint to the eval_loss that scored it. Without this
# block, the adapter actually pushed to the Hub / used for inference was just whatever
# existed at the final training step, not the best-performing one on the held-out
# validation split - a real gap, not a style choice.
save_strategy: steps
load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false
"""

# Option B target: a ~24-32GB card (originally sized for a 4090; also what fits a
# 32GB Blackwell card without falling back to QLoRA). This previously didn't match
# its own Step 4 description ("3072 context... with gradient checkpointing") - the
# generated YAML had cutoff_len=4096 and no gradient_checkpointing at all, which is
# very plausibly why a 31.36 GiB card OOM'd on the very first training step even
# with batch_size=1: full 4096-token activations, uncheckpointed, for an 8B model
# plus LoRA adapter states left too little headroom. Both are fixed here to actually
# match the documented intent - gradient_checkpointing trades some compute for a
# large activation-memory cut, the single highest-impact lever for fitting LoRA
# fine-tuning on a card this size.
yaml_content_4090 = f"""\
### model
model_name_or_path: {_base_cfg["model_name_or_path"]}

### method
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 32
lora_alpha: 16

### dataset
dataset: muspsy_task1,muspsy_task2,{_task3_dataset_name}
template: {_base_cfg["template"]}
cutoff_len: 3072
max_samples: 100000
val_size: 0.05
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: {_output_dir}
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
gradient_checkpointing: true
learning_rate: 0.0002
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
flash_attn: auto

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 500

### save (best-checkpoint selection, not just "whatever's left when training stops")
# save_steps (500, in ### output above) already matches eval_steps (500) - required for
# load_best_model_at_end: HF's Trainer needs save/eval strategy+interval to align exactly
# so it can match each saved checkpoint to the eval_loss that scored it. Without this
# block, the adapter actually pushed to the Hub / used for inference was just whatever
# existed at the final training step, not the best-performing one on the held-out
# validation split - a real gap, not a style choice.
save_strategy: steps
load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false
"""

yaml_content_qlora = f"""\
### model
model_name_or_path: {_base_cfg["model_name_or_path"]}

### method
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 32
lora_alpha: 16
quantization_bit: 4
double_quantization: true
quantization_type: nf4

### dataset
dataset: muspsy_task1,muspsy_task2,{_task3_dataset_name}
template: {_base_cfg["template"]}
cutoff_len: 4096
max_samples: 100000
val_size: 0.05
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: {_output_dir}
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
gradient_checkpointing: true
learning_rate: 0.0002
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
flash_attn: auto

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 500

### save (best-checkpoint selection, not just "whatever's left when training stops")
# save_steps (500, in ### output above) already matches eval_steps (500) - required for
# load_best_model_at_end: HF's Trainer needs save/eval strategy+interval to align exactly
# so it can match each saved checkpoint to the eval_loss that scored it. Without this
# block, the adapter actually pushed to the Hub / used for inference was just whatever
# existed at the final training step, not the best-performing one on the held-out
# validation split - a real gap, not a style choice.
save_strategy: steps
load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false
"""

# --- Hugging Face Push Configuration (Optional) ---
PUSH_TO_HUB = True                      # Set to True to upload adapters to HF automatically
# Suffixed with BASE_MODEL_CHOICE and DATASET_VARIANT so different backbone/dataset
# combinations push to separate repos instead of overwriting each other.
HF_HUB_MODEL_ID = f"thanaphatt1/{BASE_MODEL_CHOICE}-muspsy-{DATASET_VARIANT}"  # Replace username if needed
PRIVATE_REPO = True                     # Make the Hugging Face repository private or public

if PUSH_TO_HUB:
    hf_config = f"""\n
### push to hub
push_to_hub: true
hub_model_id: {HF_HUB_MODEL_ID}
hub_private_repo: {str(PRIVATE_REPO).lower()}
"""
    yaml_content_a100 += hf_config
    yaml_content_4090 += hf_config
    yaml_content_qlora += hf_config

# Choose configuration (Uncomment your choice)
# config_to_write = yaml_content_a100   # Option A: Optimized for A100 (80GB VRAM)
config_to_write = yaml_content_4090  # Option B: batch_size=1 + gradient checkpointing - fits a 24-32GB card
# config_to_write = yaml_content_qlora # Option C: 4-bit QLoRA

with open("muspsy_lora.yaml", "w", encoding="utf-8") as f:
    f.write(config_to_write)

print(f"Training configuration written to muspsy_lora.yaml (BASE_MODEL_CHOICE={BASE_MODEL_CHOICE!r}, DATASET_VARIANT={DATASET_VARIANT!r}, dataset={_task3_dataset_name!r}, output_dir={_output_dir!r}, hub_model_id={HF_HUB_MODEL_ID!r})")

## Step 5: Start Fine-Tuning
Launch the training job as a **detached background process**, not a blocking cell. A blocking `!llamafactory-cli train ...` cell ties training to this notebook's Jupyter *kernel* - if the kernel dies or restarts (browser disconnect, "Restart Kernel", the notebook server itself getting bounced), the training process is a child of that kernel and dies with it. Launching it below with `start_new_session=True` (the POSIX equivalent of `nohup ... &`) puts it in its own session instead, so it keeps running independently of this notebook for as long as the **pod itself** stays running - only stopping/terminating the pod (or the job crashing on its own) ends it.

The cell after it (**Step 5b**) checks progress by reading files back off disk (the process PID, LLaMA-Factory's own `trainer_log.jsonl`, and the raw stdout log) - it doesn't depend on any in-memory Python object from the launch cell, so it's safe to re-run anytime, including from a **completely fresh kernel** after reconnecting to the pod later.

**Important**: Since `Meta-Llama-3-8B-Instruct` is gated, and you are pushing your trained adapters to Hugging Face, you **must authenticate** by setting your Hugging Face WRITE token below.

In [ ]:
import subprocess, os

# Set your Hugging Face WRITE token to download Llama 3 and upload trained adapters.
# Get one at https://huggingface.co/settings/tokens (type "Write"). Needed regardless of
# backbone: llama3-8b requires it just to download (gated repo); every backbone needs it
# to push at the end, since PUSH_TO_HUB=True in Step 4.
os.environ["HF_TOKEN"] = "hf_your_write_token_here"

# Alternatively, uncomment the line below to log in interactively via CLI:
# !huggingface-cli login --token $HF_TOKEN

# Fail fast if the token above is still the placeholder - otherwise this wouldn't surface
# until push_to_hub fails at the very END of training (potentially hours of GPU time later),
# buried in train.log where Step 5b's tail might not even show it if earlier lines pushed it
# out of the last-20-lines window.
assert os.environ["HF_TOKEN"] != "hf_your_write_token_here", (
    "HF_TOKEN is still the placeholder string - replace it with a real Hugging Face WRITE "
    "token above before launching. Get one at https://huggingface.co/settings/tokens."
)

# Detached background launch (see Step 5 markdown above for why) - start_new_session=True
# is the key line: it puts this process in its own OS session so it survives this
# notebook's kernel dying/restarting, as long as the pod itself keeps running. stdout/
# stderr go to a log file instead of this cell's output, since nothing will be watching
# this cell live once you disconnect.
TRAIN_LOG_PATH = "train.log"
TRAIN_PID_PATH = "train.pid"

train_log_file = open(TRAIN_LOG_PATH, "w")
train_proc = subprocess.Popen(
    ["llamafactory-cli", "train", "muspsy_lora.yaml"],
    stdout=train_log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
    start_new_session=True,
)
with open(TRAIN_PID_PATH, "w") as f:
    f.write(str(train_proc.pid))

print(f"Training launched in background, PID={train_proc.pid}.")
print(f"Log file: {os.path.abspath(TRAIN_LOG_PATH)}")
print(f"PID file: {os.path.abspath(TRAIN_PID_PATH)}")
print(
    "\nThis process is now detached from this notebook's kernel - closing the notebook, "
    "losing your connection, or restarting the kernel will NOT stop it, as long as the pod "
    "itself stays running. Use the next cell (Step 5b) anytime to check progress, including "
    "from a brand-new kernel after reconnecting."
)

## Step 5b: Check training progress
Safe to re-run anytime - including from a **completely fresh kernel** after reconnecting to this pod later (e.g. after closing the notebook or a disconnect). Everything it reports is read back off disk (the process's PID, LLaMA-Factory's own `trainer_log.jsonl`, and the raw stdout log), not from the `train_proc` Python object in the cell above, which won't exist in a new kernel.

In [ ]:
import os, json

# Re-establish cwd if this is a fresh kernel that hasn't run Step 2's `%cd LLaMA-Factory` yet.
if os.path.basename(os.getcwd()) != "LLaMA-Factory" and os.path.isdir("LLaMA-Factory"):
    os.chdir("LLaMA-Factory")

TRAIN_LOG_PATH = "train.log"
TRAIN_PID_PATH = "train.pid"

# 1. Is the process still alive? (signal 0 checks existence without actually killing it)
if os.path.exists(TRAIN_PID_PATH):
    with open(TRAIN_PID_PATH) as f:
        pid = int(f.read().strip())
    try:
        os.kill(pid, 0)
        print(f"Training process (PID {pid}) is RUNNING.")
    except ProcessLookupError:
        print(f"Training process (PID {pid}) is NOT running (finished, or crashed - check the log tail below).")
else:
    print(f"No {TRAIN_PID_PATH} found in {os.getcwd()} - training hasn't been launched from here yet.")

# 2. Read output_dir back out of the config actually used to launch training (not the
#    in-memory `_output_dir` variable from Step 4, which won't exist in a fresh kernel).
output_dir = None
if os.path.exists("muspsy_lora.yaml"):
    with open("muspsy_lora.yaml") as f:
        for line in f:
            if line.startswith("output_dir:"):
                output_dir = line.split(":", 1)[1].strip()
                break

# 3. LLaMA-Factory's own progress log (trainer_log.jsonl, one JSON line per logging_steps
#    interval) is far more informative than raw stdout - current/total step, loss, percentage
#    complete, elapsed/remaining time. See train/callbacks.py's LogCallback for the schema.
#
#    Training-loss logs (every logging_steps) and eval logs (every eval_steps) are SEPARATE
#    entries in HF Trainer's log_history - LogCallback just mirrors log_history[-1], so most
#    lines in this file (the ones NOT coinciding with an eval step) have eval_loss=None simply
#    because that particular entry is a training-step log, not because eval isn't running. Scan
#    backward for the most recent line that actually HAS a non-null eval_loss, in addition to
#    the latest line, so the real validation-loss trend is visible (this is what
#    load_best_model_at_end actually selects on).
trainer_log_path = os.path.join(output_dir, "trainer_log.jsonl") if output_dir else None
if trainer_log_path and os.path.exists(trainer_log_path):
    with open(trainer_log_path) as f:
        lines = [l for l in f if l.strip()]
    if lines:
        latest = json.loads(lines[-1])
        pct = latest.get("percentage")
        pct_str = f"{pct:.1f}%" if pct is not None else "?"
        print(f"\nStep {latest.get('current_steps')}/{latest.get('total_steps')} ({pct_str} complete)")
        print(f"Epoch: {latest.get('epoch')}  Loss: {latest.get('loss')}  Eval loss: {latest.get('eval_loss')}")
        print(f"Elapsed: {latest.get('elapsed_time')}  Remaining: {latest.get('remaining_time')}")

        last_eval = None
        for line in reversed(lines):
            entry = json.loads(line)
            if entry.get("eval_loss") is not None:
                last_eval = entry
                break
        if last_eval:
            print(
                f"\nMost recent real eval: step {last_eval.get('current_steps')}, "
                f"eval_loss={last_eval.get('eval_loss')} (epoch {last_eval.get('epoch')})"
            )
        else:
            print("\nNo eval_loss recorded yet - first eval_steps interval hasn't completed.")
else:
    print(
        f"\ntrainer_log.jsonl not found yet at {trainer_log_path!r} - training may still be in its "
        f"first logging_steps interval (dataset loading/tokenization), or hasn't started yet. "
        f"Falling back to the raw log tail below."
    )

# 4. Always show the raw stdout tail too - catches things trainer_log.jsonl won't (the initial
#    dataset/tokenization phase, install-time warnings, or a crash traceback if the process died).
if os.path.exists(TRAIN_LOG_PATH):
    with open(TRAIN_LOG_PATH, encoding="utf-8", errors="replace") as f:
        tail_lines = f.readlines()[-20:]
    print(f"\n--- last 20 lines of {TRAIN_LOG_PATH} ---")
    print("".join(tail_lines))

## Step 5c: Merge the LoRA adapter into a dense checkpoint
Needed for serving Qwen3.5-9B on vLLM specifically: `--enable-lora` was confirmed (real test, `muspsy_serve_vllm_qwen35.ipynb`'s `SERVE_MODE="muspsy_unmerged"` diagnostic) to silently fail to apply the adapter on Qwen3.5's hybrid-GDN architecture ([vllm-project/vllm#49354](https://github.com/vllm-project/vllm/issues/49354)) - Task 1/Task 2 output didn't even resemble the trained format. The documented workaround is to merge the adapter into the base weights via `merge_and_unload()` (here, via LLaMA-Factory's own `llamafactory-cli export`, using the exact schema from its `examples/merge_lora/qwen3vl_lora_sft.yaml`) and serve the merged dense checkpoint directly, with no `--enable-lora` involved at all.

Not needed for `llama3-8b`/`qwen3-8b` - those serve fine via `--enable-lora` on `muspsy_serve_vllm.ipynb`'s older vLLM pin, so this step is Qwen3.5-9B-specific. Safe to skip for other backbones.

**Two ways to run this:**
1. **Same pod that trained** - reads `model_name_or_path`/`output_dir`/`template` straight out of `muspsy_lora.yaml` on disk (not the in-memory Step 4 variables), so it's safe to run from a fresh kernel even after training already finished on this pod.
2. **A different/fresh pod** (e.g. the training pod was already shut down) - `muspsy_lora.yaml` won't exist here. The cell falls back to hardcoded values matching this project's actual completed run: the adapter already lives on HF Hub at `thanaphatt1/qwen3.5-9b-muspsy-fixed` (confirmed real - this is the exact repo `muspsy_serve_vllm_qwen35.ipynb`'s `SERVE_MODE="muspsy_unmerged"` diagnostic already loaded successfully). Only needs Steps 1-2 of this notebook run first (clone + install LLaMA-Factory/torch/transformers) - training itself doesn't need to be re-run.

In [ ]:
import os

# Re-establish cwd if this is a fresh kernel that hasn't run Step 2's `%cd LLaMA-Factory` yet.
if os.path.basename(os.getcwd()) != "LLaMA-Factory" and os.path.isdir("LLaMA-Factory"):
    os.chdir("LLaMA-Factory")

# Try reading model_name_or_path/output_dir/template back out of muspsy_lora.yaml first
# (same pod that trained). Falls back to known values for this project's actual completed
# run if that file doesn't exist (a fresh pod, training pod already shut down) - the
# adapter_name_or_path becomes the HF Hub repo ID directly instead of a local path, which
# LLaMA-Factory's export step accepts exactly the same way.
_train_model_name_or_path = _train_output_dir = _train_template = None
if os.path.exists("muspsy_lora.yaml"):
    with open("muspsy_lora.yaml") as f:
        for line in f:
            if line.startswith("model_name_or_path:"):
                _train_model_name_or_path = line.split(":", 1)[1].strip()
            elif line.startswith("output_dir:"):
                _train_output_dir = line.split(":", 1)[1].strip()
            elif line.startswith("template:"):
                _train_template = line.split(":", 1)[1].strip()

if not (_train_model_name_or_path and _train_output_dir and _train_template):
    print(
        "muspsy_lora.yaml not found (or incomplete) on this pod - falling back to known "
        "values for this project's completed qwen3.5-9b run, with adapter_name_or_path "
        "pointing at the HF Hub repo directly instead of a local training-output path."
    )
    _train_model_name_or_path = "Qwen/Qwen3.5-9B"
    _train_output_dir = "thanaphatt1/qwen3.5-9b-muspsy-fixed"  # HF Hub repo, not a local path - confirmed real, already used successfully in the muspsy_unmerged diagnostic
    _train_template = "qwen3_5_nothink"

# Merged output goes in a sibling "merged" dir next to the LoRA adapter's own "lora" dir
# when running on the training pod (e.g. saves/qwen3.5-9b/lora/muspsy-fixed ->
# saves/qwen3.5-9b/merged/muspsy-fixed); on a fresh pod (Hub adapter, no "/lora/" in the
# path) it just writes to a plain local "merged_muspsy" dir instead.
if "/lora/" in _train_output_dir:
    _merged_export_dir = _train_output_dir.replace("/lora/", "/merged/")
else:
    _merged_export_dir = "merged_muspsy"

# Exact schema from LLaMA-Factory's own examples/merge_lora/qwen3vl_lora_sft.yaml (fetched
# directly from the repo, not guessed) - export_device: cpu avoids needing spare GPU memory
# for the merge itself; export_size: 5 caps shard size at 5GB per safetensors file, matching
# the documented example. trust_remote_code: true - same reason as Step 4's training config
# (Qwen3.5-9B is a VLM checkpoint).
merge_config = f"""\
### Note: DO NOT use quantized model or quantization_bit when merging lora adapters

### model
model_name_or_path: {_train_model_name_or_path}
adapter_name_or_path: {_train_output_dir}
template: {_train_template}
trust_remote_code: true

### export
export_dir: {_merged_export_dir}
export_size: 5
export_device: cpu
export_legacy_format: false
"""

with open("muspsy_merge.yaml", "w", encoding="utf-8") as f:
    f.write(merge_config)

print(f"Merge config written to muspsy_merge.yaml:\n{merge_config}")
print("Running the merge (loads full base weights + adapter, so this takes a few minutes)...")
!llamafactory-cli export muspsy_merge.yaml
print(f"\nMerged checkpoint written to: {os.path.abspath(_merged_export_dir)}")

## Step 5d: (Optional) Push the merged checkpoint to Hugging Face Hub
Only needed if you'll serve on a **different** pod than the one that trained/merged - `muspsy_serve_vllm_qwen35.ipynb`'s `MERGED_CHECKPOINT` accepts either a local path or an HF Hub repo ID. If serving happens on this same pod, just point `MERGED_CHECKPOINT` at the local `_merged_export_dir` path printed by Step 5c and skip this cell entirely.

Requires `HF_TOKEN` already set (from Step 5's launch cell, or set again here if this is a fresh kernel).

In [ ]:
import os
from huggingface_hub import HfApi

assert os.environ.get("HF_TOKEN") and os.environ["HF_TOKEN"] != "hf_your_write_token_here", (
    "HF_TOKEN not set (or still the placeholder) - set it here or re-run Step 5's token cell first."
)

# Suffixed with BASE_MODEL_CHOICE/DATASET_VARIANT, matching the adapter repo naming
# convention from Step 4 - "-merged" distinguishes it from the plain adapter repo.
MERGED_REPO_ID = f"thanaphatt1/{BASE_MODEL_CHOICE}-muspsy-{DATASET_VARIANT}-merged"  # matches muspsy_serve_vllm_qwen35.ipynb's MERGED_CHECKPOINT placeholder
MERGED_REPO_PRIVATE = True

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id=MERGED_REPO_ID, private=MERGED_REPO_PRIVATE, exist_ok=True)
api.upload_folder(
    folder_path=_merged_export_dir,
    repo_id=MERGED_REPO_ID,
    repo_type="model",
)
print(f"Merged checkpoint pushed to https://huggingface.co/{MERGED_REPO_ID}")
print(f"Set MERGED_CHECKPOINT = {MERGED_REPO_ID!r} in muspsy_serve_vllm_qwen35.ipynb's Step 2.")

## Step 6: Test the Fine-Tuned Model (Inference)
Test the newly trained model in interactive chat mode to verify its counseling capability.

In [ ]:
!llamafactory-cli chat \
    --model_name_or_path {_base_cfg["model_name_or_path"]} \
    --adapter_name_or_path {_output_dir} \
    --template {_base_cfg["template"]}